# 01 — CMS Medicare Part B: Provider-Level Claim Data

**Data source:** CMS Medicare Physician & Other Practitioners by Provider and Service  
**What it contains:** One row per provider (NPI) per procedure code (HCPCS/CPT) per year — claim counts, patient counts, Medicare allowed amounts  
**Geographic grain:** Provider ZIP code → mapped to county FIPS in cleaning step  
**DuckDB tables written:** `cms_raw`, `cms_providers`  
**Why it matters:** Identifies which providers are actively treating lymphedema and venous disease patients, ranked by volume — our primary HCP target list

### Notebook phases
1. Setup & config validation  
2. Data collection — CMS Socrata API, filtered to target codes only  
3. Cleaning — normalize columns, map ZIP → county FIPS  
4. Load to DuckDB  
5. QA & summary stats

In [24]:
import sys
import time
import pandas as pd
import duckdb
import requests
from pathlib import Path
from bs4 import BeautifulSoup

sys.path.append(str(Path('..') / 'src'))

from config import DATA_RAW, DATA_PROCESSED, DB_PATH, TARGET_CODES
from db import get_connection
from utils import ensure_dirs, log

ensure_dirs()
log("Notebook 01 — CMS Medicare — started")



[2026-04-19 18:30:26] INFO - Notebook 01 — CMS Medicare — started


# 01 — CMS Medicare Part B: Provider-Level Claim Data

**Data source:** CMS Medicare Physician & Other Practitioners by Provider and Service  
**What it contains:** One row per provider (NPI) per procedure code per year  
**Geographic grain:** Provider ZIP → mapped to county FIPS in cleaning step  
**DuckDB tables written:** `cms_raw`, `cms_providers`  
**Why it matters:** Shows which providers are actively treating lymphedema and venous 
disease patients — our primary HCP target list

In [25]:
import sys
import time
import pandas as pd
import duckdb
import requests
from pathlib import Path
from tqdm.notebook import tqdm

sys.path.append(str(Path('..') / 'src'))

from config import DATA_RAW, DATA_PROCESSED, DB_PATH, TARGET_CODES
from db import get_connection
from utils import ensure_dirs, log

ensure_dirs()
log("Notebook 01 — CMS Medicare — started")

[2026-04-19 18:30:28] INFO - Notebook 01 — CMS Medicare — started


In [26]:
# CMS Socrata API config
# Socrata is the platform CMS uses to publish public datasets
CMS_DATASET_ID = "fs4p-t5eq"   # Medicare Part B by Provider & Service, 2022
CMS_BASE_URL   = f"https://data.cms.gov/resource/{CMS_DATASET_ID}.json"
PAGE_SIZE      = 50_000        # Max rows Socrata returns per request
THROTTLE_S     = 0.5           # Seconds to wait between pages

# Only fetch columns we actually need
COLUMNS_TO_FETCH = [
    "rndrng_npi",                   # National Provider Identifier — unique ID per provider
    "rndrng_prvdr_last_org_name",   # Provider last name or organization name
    "rndrng_prvdr_first_name",      # Provider first name
    "rndrng_prvdr_type",            # Specialty (e.g. Physical Therapist, Vascular Surgery)
    "rndrng_prvdr_zip5",            # Provider ZIP code
    "rndrng_prvdr_state_abrvtn",    # State abbreviation
    "rndrng_prvdr_city",            # City
    "hcpcs_cd",                     # Procedure code billed (CPT or HCPCS)
    "hcpcs_desc",                   # Human-readable description of the code
    "tot_benes",                    # Total unique Medicare patients served
    "tot_srvcs",                    # Total claims billed
    "avg_mdcr_alowd_amt",           # Average Medicare allowed dollar amount per service
]

# Combine garment billing codes + treatment procedure codes into one list
ALL_TARGET_CODES = TARGET_CODES['HCPCS'] + TARGET_CODES['CPT']

print("API endpoint:", CMS_BASE_URL)
print("Codes to fetch:", ALL_TARGET_CODES)

API endpoint: https://data.cms.gov/resource/fs4p-t5eq.json
Codes to fetch: ['L8100', 'L8110', 'L8120', 'L8130', '97016', '97140', '97110', '93971', '36475']


In [27]:
def fetch_cms_for_code(hcpcs_code: str) -> pd.DataFrame:
    """
    Fetch all CMS Part B rows for a single HCPCS/CPT code.
    
    Paginates automatically using Socrata's $limit and $offset parameters
    until all rows for that code are retrieved.
    
    Args:
        hcpcs_code: Procedure code to query e.g. '97140', 'L8100'
    Returns:
        DataFrame with all provider rows for that code across all US states
    """
    pages  = []
    offset = 0

    while True:
        params = {
            "$where":  f"hcpcs_cd='{hcpcs_code}'",  # Filter to this code only
            "$select": ",".join(COLUMNS_TO_FETCH),   # Only columns we need
            "$limit":  PAGE_SIZE,                    # 50k rows per request
            "$offset": offset,                       # Pagination cursor
        }

        resp = requests.get(CMS_BASE_URL, params=params, timeout=30)
        resp.raise_for_status()  # Raises an error if HTTP status is 4xx/5xx
        batch = resp.json()

        if not batch:
            break  # Empty response means no more rows

        pages.append(pd.DataFrame(batch))
        offset += PAGE_SIZE

        if len(batch) < PAGE_SIZE:
            break  # Last page — fewer rows than page size means we're done

        time.sleep(THROTTLE_S)

    if not pages:
        log(f"WARNING: No data returned for {hcpcs_code}", level="WARN")
        return pd.DataFrame()

    return pd.concat(pages, ignore_index=True)

print("fetch_cms_for_code() defined")

fetch_cms_for_code() defined


In [28]:
# Updated CMS API config — new endpoint format as of 2025
DATASET_ID   = "92396110-2aed-4d63-a6a2-5d6207d46a29"
CMS_BASE_URL = f"https://data.cms.gov/data-api/v1/dataset/{DATASET_ID}/data"
PAGE_SIZE    = 5_000   # New API has lower page limits than Socrata
THROTTLE_S   = 0.5

# Test the endpoint with a small request first — just 5 rows, no filter
test_resp = requests.get(
    CMS_BASE_URL,
    params={"size": 5, "offset": 0},
    timeout=30
)

print("Status code:", test_resp.status_code)
print("Sample response:")
test_resp.json()[:2]

Status code: 200
Sample response:


[{'Rndrng_NPI': '1003000126',
  'Rndrng_Prvdr_Last_Org_Name': 'Enkeshafi',
  'Rndrng_Prvdr_First_Name': 'Ardalan',
  'Rndrng_Prvdr_MI': '',
  'Rndrng_Prvdr_Crdntls': 'M.D.',
  'Rndrng_Prvdr_Ent_Cd': 'I',
  'Rndrng_Prvdr_St1': '6410 Rockledge Dr Ste 304',
  'Rndrng_Prvdr_St2': '',
  'Rndrng_Prvdr_City': 'Bethesda',
  'Rndrng_Prvdr_State_Abrvtn': 'MD',
  'Rndrng_Prvdr_State_FIPS': '24',
  'Rndrng_Prvdr_Zip5': '20817',
  'Rndrng_Prvdr_RUCA': '1',
  'Rndrng_Prvdr_RUCA_Desc': 'Metropolitan area core: primary flow within an urbanized area of 50,000 and greater',
  'Rndrng_Prvdr_Cntry': 'US',
  'Rndrng_Prvdr_Type': 'Hospitalist',
  'Rndrng_Prvdr_Mdcr_Prtcptg_Ind': 'Y',
  'HCPCS_Cd': '99221',
  'HCPCS_Desc': 'Initial hospital care with straightforward or low level of medical decision making, per day, if using time, at least 40 minutes',
  'HCPCS_Drug_Ind': 'N',
  'Place_Of_Srvc': 'F',
  'Tot_Benes': '12',
  'Tot_Srvcs': '12',
  'Tot_Bene_Day_Srvcs': '12',
  'Avg_Sbmtd_Chrg': '250.22666667',
  

In [29]:
# Updated column names to match new API response (Title_Case)
COLUMNS_TO_FETCH = [
    "Rndrng_NPI",
    "Rndrng_Prvdr_Last_Org_Name",
    "Rndrng_Prvdr_First_Name",
    "Rndrng_Prvdr_Type",
    "Rndrng_Prvdr_City",
    "Rndrng_Prvdr_State_Abrvtn",
    "Rndrng_Prvdr_State_FIPS",
    "Rndrng_Prvdr_Zip5",
    "HCPCS_Cd",
    "HCPCS_Desc",
    "Tot_Benes",
    "Tot_Srvcs",
    "Avg_Mdcr_Alowd_Amt",
]

def fetch_cms_for_code(hcpcs_code: str) -> pd.DataFrame:
    """
    Fetch all CMS Part B rows for a single HCPCS/CPT code.
    Uses the new CMS data-api/v1 endpoint (replaced Socrata in 2025).
    Paginates until all rows for that code are retrieved.

    Args:
        hcpcs_code: Procedure code to query e.g. '97140', 'L8100'
    Returns:
        DataFrame with all provider rows for that code across all US states
    """
    pages  = []
    offset = 0

    while True:
        params = {
            "filter[HCPCS_Cd]": hcpcs_code,  # New API uses filter[] syntax
            "size":   PAGE_SIZE,
            "offset": offset,
        }

        resp = requests.get(CMS_BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        batch = resp.json()

        if not batch:
            break

        # Keep only columns we need
        df_batch = pd.DataFrame(batch)[COLUMNS_TO_FETCH]
        pages.append(df_batch)
        offset += PAGE_SIZE

        if len(batch) < PAGE_SIZE:
            break  # Last page

        time.sleep(THROTTLE_S)

    if not pages:
        log(f"WARNING: No data returned for {hcpcs_code}", level="WARN")
        return pd.DataFrame()

    return pd.concat(pages, ignore_index=True)

print("fetch_cms_for_code() updated for new API")

fetch_cms_for_code() updated for new API


In [30]:
# Test with 97140 — Manual lymphatic drainage (MLD)
# Highest value CPT code — therapists billing this are our #1 target
log("Testing fetch for code 97140...")

test_df = fetch_cms_for_code("97140")

print(f"Rows returned: {len(test_df):,}")
print(f"Columns: {test_df.columns.tolist()}")
test_df.head(3)

[2026-04-19 18:30:33] INFO - Testing fetch for code 97140...
Rows returned: 65,139
Columns: ['Rndrng_NPI', 'Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'HCPCS_Cd', 'HCPCS_Desc', 'Tot_Benes', 'Tot_Srvcs', 'Avg_Mdcr_Alowd_Amt']


,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_Type,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_Zip5,HCPCS_Cd,HCPCS_Desc,Tot_Benes,Tot_Srvcs,Avg_Mdcr_Alowd_Amt
0,1003000829,Kochanek,Michelle,Physical Therapist in Private Practice,Denver,CO,08,80222,97140,"Therapy procedure using manual technique, each...",55,570,22.619596491
1,1003001140,Nicastro,Jon,Physical Therapist in Private Practice,Henderson,NV,32,89014,97140,"Therapy procedure using manual technique, each...",123,627,20.074593301
2,1003001249,Day,Ronnie,Physical Therapist in Private Practice,Danville,IL,17,61832,97140,"Therapy procedure using manual technique, each...",19,115,20.732695652


In [31]:
# Fetch all target codes and combine into one DataFrame
# HCPCS = compression garment billing codes (L8100 etc.)
# CPT   = treatment procedure codes (97140 etc.)

all_frames = []

for i, code in enumerate(ALL_TARGET_CODES):
    log(f"[{i+1}/{len(ALL_TARGET_CODES)}] Fetching code: {code}")
    df = fetch_cms_for_code(code)
    if not df.empty:
        all_frames.append(df)
        log(f"  → {len(df):,} rows for {code}")
    time.sleep(THROTTLE_S)

cms_raw = pd.concat(all_frames, ignore_index=True)

log(f"Total rows collected: {len(cms_raw):,}")
log(f"Unique providers (NPIs): {cms_raw['Rndrng_NPI'].nunique():,}")
log(f"Codes represented: {cms_raw['HCPCS_Cd'].unique().tolist()}")

[2026-04-19 18:30:52] INFO - [1/9] Fetching code: L8100
[2026-04-19 18:30:52] WARN - WARNING: No data returned for L8100
[2026-04-19 18:30:53] INFO - [2/9] Fetching code: L8110
[2026-04-19 18:30:54] WARN - WARNING: No data returned for L8110
[2026-04-19 18:30:54] INFO - [3/9] Fetching code: L8120
[2026-04-19 18:30:55] WARN - WARNING: No data returned for L8120
[2026-04-19 18:30:56] INFO - [4/9] Fetching code: L8130
[2026-04-19 18:30:57] WARN - WARNING: No data returned for L8130
[2026-04-19 18:30:57] INFO - [5/9] Fetching code: 97016
[2026-04-19 18:31:17] INFO -   → 4,370 rows for 97016
[2026-04-19 18:31:17] INFO - [6/9] Fetching code: 97140
[2026-04-19 18:31:34] INFO -   → 65,139 rows for 97140
[2026-04-19 18:31:35] INFO - [7/9] Fetching code: 97110
[2026-04-19 18:33:57] INFO -   → 80,024 rows for 97110
[2026-04-19 18:33:58] INFO - [8/9] Fetching code: 93971
[2026-04-19 18:35:19] INFO -   → 24,534 rows for 93971
[2026-04-19 18:35:20] INFO - [9/9] Fetching code: 36475
[2026-04-19 18:35

In [34]:
# Save raw pull to disk before any cleaning
# Using CSV instead of parquet due to pyarrow compatibility with pandas 3.0
raw_out = DATA_RAW['cms'] / 'cms_partb_2023_raw.csv'
cms_raw.to_csv(raw_out, index=False)

log(f"Raw data saved to: {raw_out}")
log(f"File size: {raw_out.stat().st_size / 1_048_576:.1f} MB")


[2026-04-19 18:40:09] INFO - Raw data saved to: C:\Users\sanat\Desktop\AntiGravity\findingclaims\data\raw\cms\cms_partb_2023_raw.csv
[2026-04-19 18:40:09] INFO - File size: 33.1 MB


In [35]:
# ── Step 2: Clean the raw CMS data ─────────────────────────────────────────
cms = cms_raw.copy()

# 1. Cast numerics — API returns everything as strings
numeric_cols = ['Tot_Benes', 'Tot_Srvcs', 'Avg_Mdcr_Alowd_Amt']
for col in numeric_cols:
    cms[col] = pd.to_numeric(cms[col], errors='coerce')

# 2. Normalize ZIP to 5-digit string
# Some ZIPs lose leading zero e.g. "1234" → "01234"
cms['zip5'] = (
    cms['Rndrng_Prvdr_Zip5']
    .astype(str)
    .str.strip()
    .str.zfill(5)
    .str[:5]
)

# 3. Drop rows missing NPI or invalid ZIP — can't place them geographically
before = len(cms)
cms = cms.dropna(subset=['Rndrng_NPI', 'zip5'])
cms = cms[cms['zip5'].str.match(r'^\d{5}$')]
log(f"Dropped {before - len(cms):,} rows missing NPI or valid ZIP")

# 4. Add code_type label — helps split garment billing vs treatment procedures
cms['code_type'] = cms['HCPCS_Cd'].apply(
    lambda c: 'HCPCS_garment' if c in TARGET_CODES['HCPCS'] else 'CPT_procedure'
)

# 5. Combine first and last name into one clean field
cms['provider_name'] = (
    cms['Rndrng_Prvdr_First_Name'].fillna('') + ' ' +
    cms['Rndrng_Prvdr_Last_Org_Name'].fillna('')
).str.strip()

log(f"Rows after cleaning: {len(cms):,}")
log(f"Unique NPIs: {cms['Rndrng_NPI'].nunique():,}")
cms.head(3)

[2026-04-19 18:40:54] INFO - Dropped 0 rows missing NPI or valid ZIP
[2026-04-19 18:40:54] INFO - Rows after cleaning: 175,290
[2026-04-19 18:40:55] INFO - Unique NPIs: 104,360


,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_Type,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_Zip5,HCPCS_Cd,HCPCS_Desc,Tot_Benes,Tot_Srvcs,Avg_Mdcr_Alowd_Amt,zip5,code_type,provider_name
0,1003096884,Rarich,Jason,Physical Therapist in Private Practice,Collegeville,PA,42,19426,97016,Application of blood vessel compression device,45,212.0,8.872453,19426,CPT_procedure,Jason Rarich
1,1003126202,Conradi,Robin,Physical Therapist in Private Practice,Durham,NC,37,27704,97016,Application of blood vessel compression device,28,286.0,8.879895,27704,CPT_procedure,Robin Conradi
2,1003153800,Gill,Bikramjit,Physician Assistant,Mesa,AZ,04,85209,97016,Application of blood vessel compression device,146,1472.0,7.533125,85209,CPT_procedure,Bikramjit Gill


In [36]:
# ── Map ZIP → county FIPS ───────────────────────────────────────────────────
# CMS data has ZIP codes but our geographic grain is county (FIPS code).
# FIPS = Federal Information Processing Standards — a 5-digit code where
# first 2 digits = state, last 3 digits = county e.g. "06037" = Los Angeles County CA
#
# We use the Census ZCTA-to-county relationship file — no API key needed,
# annually updated, covers all US ZIPs

CENSUS_ZIP_COUNTY_URL = (
    "https://www2.census.gov/geo/docs/maps-data/data/rel2020/"
    "zcta520/tab20_zcta520_county20_natl.txt"
)

log("Downloading Census ZIP-to-county crosswalk...")
zcta_county = pd.read_csv(
    CENSUS_ZIP_COUNTY_URL,
    sep='|',
    dtype=str,
)

log(f"Crosswalk columns: {zcta_county.columns.tolist()}")
zcta_county.head(3)

[2026-04-19 18:44:09] INFO - Downloading Census ZIP-to-county crosswalk...
[2026-04-19 18:44:10] INFO - Crosswalk columns: ['OID_ZCTA5_20', 'GEOID_ZCTA5_20', 'NAMELSAD_ZCTA5_20', 'AREALAND_ZCTA5_20', 'AREAWATER_ZCTA5_20', 'MTFCC_ZCTA5_20', 'CLASSFP_ZCTA5_20', 'FUNCSTAT_ZCTA5_20', 'OID_COUNTY_20', 'GEOID_COUNTY_20', 'NAMELSAD_COUNTY_20', 'AREALAND_COUNTY_20', 'AREAWATER_COUNTY_20', 'MTFCC_COUNTY_20', 'CLASSFP_COUNTY_20', 'FUNCSTAT_COUNTY_20', 'AREALAND_PART', 'AREAWATER_PART']


,OID_ZCTA5_20,GEOID_ZCTA5_20,NAMELSAD_ZCTA5_20,AREALAND_ZCTA5_20,AREAWATER_ZCTA5_20,MTFCC_ZCTA5_20,CLASSFP_ZCTA5_20,FUNCSTAT_ZCTA5_20,OID_COUNTY_20,GEOID_COUNTY_20,NAMELSAD_COUNTY_20,AREALAND_COUNTY_20,AREAWATER_COUNTY_20,MTFCC_COUNTY_20,CLASSFP_COUNTY_20,FUNCSTAT_COUNTY_20,AREALAND_PART,AREAWATER_PART
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27590114112812,01003,Baldwin County,4117656199,1132956041,G4020,H1,A,339765765,927218265
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2759099719300,01007,Bibb County,1612188717,9572303,G4020,H1,A,92709500,11113
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27590103020886,01015,Calhoun County,1569246127,16536293,G4020,H1,A,173314495,461654


In [37]:
# Keep only the columns we need from the crosswalk
zcta_county = zcta_county[['GEOID_ZCTA5_20', 'GEOID_COUNTY_20', 'NAMELSAD_COUNTY_20']].rename(columns={
    'GEOID_ZCTA5_20':   'zip5',
    'GEOID_COUNTY_20':  'fips_county',   # 5-digit FIPS e.g. "06037" = Los Angeles CA
    'NAMELSAD_COUNTY_20': 'county_name', # Human readable e.g. "Los Angeles County"
})

# Some ZIPs span multiple counties — keep the first match only
# (Census orders by largest land area overlap — good enough for our use case)
zcta_county = zcta_county.drop_duplicates(subset='zip5', keep='first')

log(f"Crosswalk ready: {len(zcta_county):,} ZIP-to-county mappings")

# Join onto CMS data
cms = cms.merge(zcta_county, on='zip5', how='left')

# Check how many rows matched
matched   = cms['fips_county'].notna().sum()
unmatched = cms['fips_county'].isna().sum()
log(f"Matched:   {matched:,} rows ({matched/len(cms)*100:.1f}%)")
log(f"Unmatched: {unmatched:,} rows ({unmatched/len(cms)*100:.1f}%) — PO boxes, military ZIPs, territories")

cms[['provider_name', 'zip5', 'fips_county', 'county_name']].head(5)

[2026-04-19 18:44:37] INFO - Crosswalk ready: 33,792 ZIP-to-county mappings
[2026-04-19 18:44:37] INFO - Matched:   174,332 rows (99.5%)
[2026-04-19 18:44:37] INFO - Unmatched: 958 rows (0.5%) — PO boxes, military ZIPs, territories


,provider_name,zip5,fips_county,county_name
0,Jason Rarich,19426,42091,Montgomery County
1,Robin Conradi,27704,37063,Durham County
2,Bikramjit Gill,85209,04013,Maricopa County
3,Andrew Chen,12561,36111,Ulster County
4,Shawn Moody,27612,37183,Wake County


In [38]:
# ── Build provider-level summary ────────────────────────────────────────────
# Roll up from one-row-per-NPI-per-code to one-row-per-NPI
# This is our main targeting table — one provider per row, ranked by patient volume

cms_providers = (
    cms.groupby([
        'Rndrng_NPI',
        'provider_name',
        'Rndrng_Prvdr_Type',        # Specialty e.g. "Physical Therapist in Private Practice"
        'Rndrng_Prvdr_City',
        'Rndrng_Prvdr_State_Abrvtn',
        'Rndrng_Prvdr_State_FIPS',  # 2-digit state FIPS e.g. "06" = California
        'zip5',
        'fips_county',              # 5-digit county FIPS — our geographic join key
        'county_name',
    ])
    .agg(
        total_patients    = ('Tot_Benes', 'sum'),        # Total unique Medicare patients
        total_claims      = ('Tot_Srvcs', 'sum'),        # Total claims billed
        avg_allowed_amt   = ('Avg_Mdcr_Alowd_Amt', 'mean'), # Avg Medicare allowed $ 
        codes_billed      = ('HCPCS_Cd', lambda x: ','.join(sorted(x.unique()))),
        n_target_codes    = ('HCPCS_Cd', 'nunique'),    # How many of our 9 codes they bill
    )
    .reset_index()
    .sort_values('total_patients', ascending=False)
)

log(f"Provider summary: {len(cms_providers):,} unique providers")
log(f"Top specialties:")
print(
    cms_providers
    .groupby('Rndrng_Prvdr_Type')['Rndrng_NPI']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

[2026-04-19 18:45:31] INFO - Provider summary: 103,621 unique providers
[2026-04-19 18:45:31] INFO - Top specialties:
Rndrng_Prvdr_Type
Physical Therapist in Private Practice            70937
Diagnostic Radiology                              14177
Occupational Therapist in Private Practice         9165
Vascular Surgery                                   2279
Cardiology                                         1167
Orthopedic Surgery                                  853
Interventional Radiology                            801
General Surgery                                     562
Independent Diagnostic Testing Facility (IDTF)      519
Interventional Cardiology                           454


In [39]:
# ── Write to DuckDB ─────────────────────────────────────────────────────────
con = get_connection()

# Write cms_raw — granular, one row per NPI per code
con.execute("DROP TABLE IF EXISTS cms_raw")
con.execute("CREATE TABLE cms_raw AS SELECT * FROM cms")
count_raw = con.execute("SELECT COUNT(*) FROM cms_raw").fetchone()[0]
log(f"cms_raw written: {count_raw:,} rows")

# Write cms_providers — one row per NPI, rolled up
con.execute("DROP TABLE IF EXISTS cms_providers")
con.execute("CREATE TABLE cms_providers AS SELECT * FROM cms_providers")
count_prov = con.execute("SELECT COUNT(*) FROM cms_providers").fetchone()[0]
log(f"cms_providers written: {count_prov:,} rows")

con.close()
log("DuckDB connection closed")

[2026-04-19 18:46:54] INFO - cms_raw written: 175,290 rows
[2026-04-19 18:46:54] INFO - cms_providers written: 103,621 rows
[2026-04-19 18:46:54] INFO - DuckDB connection closed


In [40]:
# ── Save processed files to disk ────────────────────────────────────────────
processed_raw  = DATA_PROCESSED['cms'] / 'cms_partb_2023_clean.csv'
processed_prov = DATA_PROCESSED['cms'] / 'cms_providers_2023.csv'

cms.to_csv(processed_raw, index=False)
cms_providers.to_csv(processed_prov, index=False)

log(f"Clean granular data → {processed_raw}")
log(f"Provider summary    → {processed_prov}")
log(f"Sizes: {processed_raw.stat().st_size / 1_048_576:.1f} MB granular, "
    f"{processed_prov.stat().st_size / 1_048_576:.1f} MB providers")

[2026-04-19 18:47:32] INFO - Clean granular data → C:\Users\sanat\Desktop\AntiGravity\findingclaims\data\processed\cms\cms_partb_2023_clean.csv
[2026-04-19 18:47:32] INFO - Provider summary    → C:\Users\sanat\Desktop\AntiGravity\findingclaims\data\processed\cms\cms_providers_2023.csv
[2026-04-19 18:47:32] INFO - Sizes: 42.8 MB granular, 14.0 MB providers


In [41]:
# ── QA & summary stats ──────────────────────────────────────────────────────
con = get_connection()

print("=== Volume by code ===")
print(con.execute("""
    SELECT 
        HCPCS_Cd                                    AS code,
        HCPCS_Desc                                  AS description,
        COUNT(DISTINCT Rndrng_NPI)                  AS providers,
        SUM(Tot_Benes)                              AS medicare_patients
    FROM cms_raw
    GROUP BY HCPCS_Cd, HCPCS_Desc
    ORDER BY medicare_patients DESC
""").df().to_string(index=False))

print("\n=== Top 10 states by provider count ===")
print(con.execute("""
    SELECT 
        Rndrng_Prvdr_State_Abrvtn   AS state,
        COUNT(*)                    AS providers,
        SUM(total_patients)         AS medicare_patients
    FROM cms_providers
    GROUP BY state
    ORDER BY providers DESC
    LIMIT 10
""").df().to_string(index=False))

print("\n=== Providers billing 2+ of our target codes (highest value targets) ===")
print(con.execute("""
    SELECT 
        n_target_codes,
        COUNT(*) AS providers
    FROM cms_providers
    GROUP BY n_target_codes
    ORDER BY n_target_codes DESC
""").df().to_string(index=False))

con.close()

=== Volume by code ===
 code                                                                                                        description  providers  medicare_patients
97110 Therapy procedure using exercise to develop strength, endurance, range of motion, and flexibility, each 15 minutes      79999          4977286.0
97140                                                          Therapy procedure using manual technique, each 15 minutes      65133          3310126.0
93971                                            Ultrasound study of one arm or leg veins with compression and maneuvers      21388          1269628.0
97016                                                                     Application of blood vessel compression device       4370           139735.0
36475                      Destruction of first incompetent vein of arm or leg using radiofrequency and imaging guidance       1221            36461.0

=== Top 10 states by provider count ===
state  providers  medicare_pat

Highlights:

- 97110 and 97140 together cover 145k providers and 8.2M Medicare patients — your addressable market just from these two codes alone is massive
- CA, NY, FL, TX, NJ are your top 5 states by provider count — expected for a national launch, prioritize these territories first
- 59,771 providers billing 2+ of our target codes — these are your Tier 1 targets, they're already treating multiple relevant conditions
- 3,917 billing 3+ codes — these are your highest priority accounts, worth white-glove outreach

## Notebook 01 complete ✓

### What we built
- Pulled 175,290 rows from CMS Medicare Part B 2023 via API — filtered to 9 target codes
- Mapped provider ZIPs to county FIPS (99.5% match rate)
- Built `cms_providers` — 103,621 unique providers ranked by Medicare patient volume
- Written to DuckDB tables: `cms_raw`, `cms_providers`
- Saved to `data/processed/cms/`

### Key findings
- 70,937 Physical Therapists billing manual lymphatic drainage (CPT 97140) — core target
- 9,165 Occupational Therapists — secondary target
- 2,279 Vascular Surgeons — post-ablation compression garment prescribers
- Top states: CA, NY, FL, TX, NJ
- 59,771 providers billing 2+ target codes — Tier 1 priority accounts

### Limitations
- Medicare only (~35% of total patient population)
- L-codes (compression garment billing) not in this dataset — live in CMS DMEPOS dataset
- 2023 data — ~18 month lag from claim year to public release

### Next notebook
`02_seer_cancer.ipynb` — county-level cancer incidence data  
Gives us geographic demand signal for the commercially-insured population  
that Medicare cannot see. Breast cancer survivorship drives ~60% of secondary lymphedema.